In [43]:
from standard import *
import plotly.express as px
rng = np.random.RandomState(24)

In [44]:
df = pd.read_csv('./data/raw/DK1 load.csv', parse_dates=['utc_timestamp'])
df['utc_timestamp'] = df['utc_timestamp'].dt.tz_localize(None)
df['date'] = df['utc_timestamp'].dt.normalize()
df['hour'] = df['utc_timestamp'].dt.hour
df['dow'] = df['utc_timestamp'].dt.day_of_week
df['month'] = df['utc_timestamp'].dt.month
df['year'] = df['utc_timestamp'].dt.year
df['schedule'] = df.groupby(['date']).ngroup()
df['load'] = df['DK_1_load_actual_entsoe_transparency']
df['schedule_hour'] = df['schedule'].astype(str) + '_' + df['hour'].astype(str)
df = df[['schedule','hour','schedule_hour','load']]
df = df[df['load'].notna()].copy()
load_data = df.copy()

# Generators setup

class Generator():
    def __init__(self, gen_type, p_min,p_max,mc,sc,lock_time,ramp_hour,online,locked,prev_dispatch=None):
        
        # Nameplate info
        self.id = 0
        self.gen_type = gen_type
        self.p_min = p_min
        self.p_max = p_max
        self.mc = mc
        self.sc = sc
        self.lock_time = lock_time
        self.ramp_hour = ramp_hour
        
        # Current state
        self.online = online
        self.locked = locked
        if self.online:
            self.locked = False
        
        if prev_dispatch == None:
            self.prev_dispatch = p_min
            
gas = Generator('gas',p_min=1,p_max=50,mc=100,sc=50,lock_time=1,ramp_hour=100,online=True,locked=False)
coal = Generator('coal',p_min=10,p_max=200,mc=80,sc=1600,lock_time=4,ramp_hour=50,online=True,locked=False)
hydro = Generator('hydro',p_min=0,p_max=200,mc=40,sc=5,lock_time=0,ramp_hour=200,online=True,locked=False)
nuclear = Generator('nuclear',p_min=500,p_max=1000,mc=10,sc=5,lock_time=72,ramp_hour=600,online=True,locked=False)

ratios = np.array([20,12,3,3]) # gas, coal, hydro, nuclear
p_maxes = np.array([50,200,200,1000])
print('Max load: ',df['load'].max())
print('Max capacity: ',np.sum(ratios * p_maxes))
np.sum(ratios * p_maxes) >= df['load'].max()

gas_gens = [
    Generator('gas',p_min=1,p_max=50,mc=rng.randint(90,100),sc=50,lock_time=1,ramp_hour=100,online=True,locked=False) 
    for _ in range(20)
    ]

coal_gens = [
    Generator('coal',p_min=10,p_max=200,mc=rng.randint(60,90),sc=1600,lock_time=4,ramp_hour=50,online=True,locked=False) 
    for _ in range(12)
    ]

hydro_gens = [
    Generator('hydro',p_min=0,p_max=200,mc=rng.randint(20,30),sc=5,lock_time=0,ramp_hour=200,online=True,locked=False)
    for _ in range(2)
    ]

nuclear_gens = [
    Generator('nuclear',p_min=500,p_max=1000,mc=rng.randint(10,50),sc=5,lock_time=72,ramp_hour=600,online=True,locked=False)
    for _ in range(2)
    ]

gas_bids = pd.DataFrame([
    {
        'p_max': gen.p_max,
        'p_min': gen.p_min,
        'mc': gen.mc,
        'sc': gen.sc,
        'lock_time': gen.lock_time,
        'ramp_hour': gen.ramp_hour,
        'online': gen.online,
        'locked': gen.locked,
        'prev_dispatch': gen.prev_dispatch,
        'max_cap': min(gen.prev_dispatch + gen.ramp_hour, gen.p_max),
        'min_cap': max(gen.prev_dispatch - gen.ramp_hour, gen.p_min),
        'type': gen.gen_type,
        'dispatch': gen.prev_dispatch
    }
    for gen in gas_gens
])

coal_bids = pd.DataFrame([
    {
        'p_max': gen.p_max,
        'p_min': gen.p_min,
        'mc': gen.mc,
        'sc': gen.sc,
        'lock_time': gen.lock_time,
        'ramp_hour': gen.ramp_hour,
        'online': gen.online,
        'locked': gen.locked,
        'prev_dispatch': gen.prev_dispatch,
        'max_cap': min(gen.prev_dispatch + gen.ramp_hour, gen.p_max),
        'min_cap': max(gen.prev_dispatch - gen.ramp_hour, gen.p_min),
        'type': gen.gen_type,
        'dispatch': gen.prev_dispatch
    }
    for gen in coal_gens
])

hydro_bids = pd.DataFrame([
    {
        'p_max': gen.p_max,
        'p_min': gen.p_min,
        'mc': gen.mc,
        'sc': gen.sc,
        'lock_time': gen.lock_time,
        'ramp_hour': gen.ramp_hour,
        'online': gen.online,
        'locked': gen.locked,
        'prev_dispatch': gen.prev_dispatch,
        'max_cap': min(gen.prev_dispatch + gen.ramp_hour, gen.p_max),
        'min_cap': max(gen.prev_dispatch - gen.ramp_hour, gen.p_min),
        'type': gen.gen_type,
        'dispatch': gen.prev_dispatch
    }
    for gen in hydro_gens
])

nuclear_bids = pd.DataFrame([
    {
        'p_max': gen.p_max,
        'p_min': gen.p_min,
        'mc': gen.mc,
        'sc': gen.sc,
        'lock_time': gen.lock_time,
        'ramp_hour': gen.ramp_hour,
        'online': gen.online,
        'locked': gen.locked,
        'prev_dispatch': gen.prev_dispatch,
        'max_cap': min(gen.prev_dispatch + gen.ramp_hour, gen.p_max),
        'min_cap': max(gen.prev_dispatch - gen.ramp_hour, gen.p_min),
        'type': gen.gen_type,
        'dispatch': gen.prev_dispatch,

    }
    for gen in nuclear_gens
])

bids = pd.DataFrame()
bids = pd.concat([gas_bids, coal_bids, hydro_bids, nuclear_bids])
bids = bids.reset_index(drop=True)
bids['id'] = bids.index
bids = bids[[bids.columns[-1], *bids.columns[:-1]]].copy()
original_bids = bids.copy()

Max load:  6711.91
Max capacity:  7000


In [ ]:
# New Market Model

class Market():
    def __init__(self, initial_bids_state:pd.DataFrame, load_schedule:pd.DataFrame):
        self.bids = initial_bids_state
        self.load_schedule = load_schedule
        self.logs = pd.DataFrame()
        self.market_logs = pd.DataFrame()
        self.market_log_slots = None
        self.undersupply = False
        
        # to normalize data type
        self.bids["locked"] = self.bids["locked"].astype("int32")
        self.bids["lock_time"] = self.bids["lock_time"].astype("int32")
        
    def start(self):
        
        # Main loop over each hour begins here
        
        for row in self.load_schedule.itertuples(index=False):
            self.load = row.load
            self.schedule = row.schedule
            self.hour = row.hour
            self.shutdown_costs = 0
            self.startup_costs = 0
            self.energy_costs = 0
            
            self.event_squence = 0
            self.logging = False
            if self.market_log_slots != None:
                if (self.schedule, self.hour) in self.market_log_slots:
                    self.logging = True
            
            # Enter into the market cycle
            self.__market_cycle()
            if self.undersupply:
                print('Load exceeds capacity!')
                return
            self.__startup_units()
            self.__calculate_cost_price()
            self.__record_logs() 
    
    def __prepare_new_bids(self):
        b = self.bids_now.copy()
        b["prev_dispatch"] = np.where(b["dispatch"] > 0, b["dispatch"], b["p_min"])
        b["max_cap"] = np.where(
            b["prev_dispatch"] + b["ramp_hour"] <= b["p_max"],
            b["prev_dispatch"] + b["ramp_hour"],
            b["p_max"],
        )
        b["min_cap"] = np.where(
            b["prev_dispatch"] - b["ramp_hour"] >= b["p_min"],
            b["prev_dispatch"] - b["ramp_hour"],
            b["p_min"],
        )        
        self.bids_now = b.copy()
    
    def activate_market_logs(self, from_sch_hour:tuple, to_sch_hour:tuple):
        f = from_sch_hour
        t = to_sch_hour
        market_log_slots = []
        for schedule in range(f[0], t[0] + 1):
            start_hour = f[1] if schedule == f[0] else 0
            end_hour = t[1] if schedule == t[0] else 23
            for hour in range(start_hour, end_hour + 1):
                market_log_slots.append((schedule, hour))
        self.market_log_slots = market_log_slots
        
    def __log_market_log(self, event_label='',final=False):
        if not self.logging:
            return
        
        b = self.bids_now.copy()
        b['event'] = event_label
        b['sequence'] = self.event_squence
        b['schedule'] = self.schedule
        b['hour'] = self.hour
        self.event_squence += 1
        self.market_logs = pd.concat([self.market_logs, b])

    def __market_cycle(self):
        
        self.bids_now = self.bids.copy()
        self.__log_market_log('start')
        
        self.__prepare_new_bids()        
        self.__update_locks()
        self.__order_by_merit()
        if self.undersupply:
            return
        self.__check_constraints()
        
        # market operator / system operator must 
        # decide what to do with constraints
        
        # 1. shutdown non-preferred unit
        if self.constraints_found:   
            self.__shutdown_nonpreferred(preferred_ids=[]) # try 35,34
            self.__order_by_merit()
            if self.undersupply:
                return
            self.__check_constraints()
            

            # 2. curtail in favor of marginal unit
            if self.constraints_found:
                self.__curtailment_protocol()
                
                # 3. if that's not feasible, shutdown marginal until no more constraint
                if not self.curtailment_feasible:
                    print(self.curtailment_feasible)
                    while self.constraints_found:
                        self.__shutdown_marginal_only()
                        self.__order_by_merit()
                        if self.undersupply:
                            return
                        self.__check_constraints()

        # clean up temporary columns
        self.bids_now = self.bids_now.drop(columns=['constrained', 'curtail', 'redispatch'], errors='ignore')
  
    def __update_locks(self):
        b = self.bids_now.copy()
        b["locked"] = np.where(b["locked"] - 1 >= 0, b["locked"] - 1, b["locked"])
        self.bids_now = b
        self.__log_market_log('updating_locks')
        
    def __order_by_merit(self):
        load = self.load
        b = self.bids_now.copy()
        b = b.sort_values(["locked", "mc"])

        self.undersupply = False

        # Check if load exceeds available capacity
        # If so, break the market cycle
        
        if load > b[b["locked"] == 0]["max_cap"].sum():
            self.undersupply = True
            return

        dispatches = []
        leftovers = []

        leftover = load
        for row in b.itertuples(index=False):
            dispatch = min(leftover, row.max_cap)
            dispatches.append(dispatch)
            leftover = leftover - dispatch
            leftovers.append(leftover)

            if leftover <= 0:
                pass

        b["dispatch"] = dispatches
        b["leftover"] = leftovers
                
        self.bids_now = b
        self.__log_market_log('merit_ordering')
    
    def __check_constraints(self):
        b = self.bids_now.copy()
        active_cond = b["dispatch"] > 0
        b["constrained"] = 0
        b["constrained"] = np.where(
            (b["dispatch"] < b["p_min"]) & active_cond, 1, b["constrained"]
        )
        b["constrained"] = np.where(
            (b["dispatch"] > b["p_max"]) & active_cond, 1, b["constrained"]
        )
        b["constrained"] = np.where((b["locked"] > 0) & active_cond, 1, b["constrained"])
        b["constrained"] = np.where(
            (b["online"] == True) & ~(active_cond), 1, b["constrained"]
        )
        b['constrained'] = np.where(
            b["online"] == False, 0, b["constrained"]
        )
        
        if b["constrained"].sum() > 0:
            self.constraints_found = True
        else:
            self.constraints_found = False
            
        self.bids_now = b.copy()
        self.__log_market_log('checking_constraint')

    def __shutdown_nonpreferred(self, preferred_ids=[]):
        
        b = self.bids_now.copy()
        mask = (b['constrained'] == 1) & (~b['id'].isin(preferred_ids) & (b['dispatch'] == 0))
        b.loc[mask, "online"] = False
        b.loc[mask, "locked"] = b.loc[mask, "lock_time"]
        b.loc[mask, "prev_dispatch"] = b.loc[mask, "p_min"]
        b.loc[mask, "min_cap"] = b.loc[mask, "p_min"]
        b.loc[mask, "max_cap"] = b.loc[mask, "prev_dispatch"] + b.loc[mask, "ramp_hour"]
        b['max_cap'] = np.where(b['max_cap'] > b['p_max'], b['p_max'], b['max_cap'])
        shutdown_cost = b[mask]["sc"].sum()
        self.shutdown_costs += shutdown_cost
        self.bids_now = b.copy()
        self.__log_market_log('shutting_down_nonpreferred')

    def __curtailment_protocol(self):
        load = self.load
        b = self.bids_now.copy()
        b["curtail"] = np.where((b["dispatch"] > 0) & (b["constrained"] == 0), 1, 0)
        retained_power = b[b["constrained"] == 1]["p_min"].sum()
        final_curtailed_power = load - retained_power
        initial_power_to_curtail = b[b["curtail"] == 1]["dispatch"].sum()
        if initial_power_to_curtail != 0:
            curtailment_factor = final_curtailed_power / initial_power_to_curtail
        else:
            curtailment_factor = 0
            
        b["redispatch"] = np.where(
            b["dispatch"] * curtailment_factor >= b["p_min"],
            b["dispatch"] * curtailment_factor,
            b["p_min"],
        )
        b["redispatch"] = np.where(
            (b["constrained"] == 0) & (b["curtail"] == 0), 0, b["redispatch"]
        )
        
        self.curtailment_feasible = None

        total_redispatch = b["redispatch"].sum()
        tol = 1e-6  # prevents floating point comparison errors e.g. 1751.4699999999998 < 1751.47
        
        if total_redispatch + tol >= load :
            b["dispatch"] = b["redispatch"]
            self.curtailment_feasible = True
            self.bids_now = b.copy()
        else:
            b["dispatch"] = b["redispatch"]
            self.curtailment_feasible = False
            self.bids_now = b.copy()
            
        self.__log_market_log('attempting_curtailment')

    def __shutdown_marginal_only(self):
        b = self.bids_now.copy()
        mask = (b['online'] == True) & (b['online'].shift(-1) == False)
        b.loc[mask, "online"] = False
        b.loc[mask, "locked"] = b.loc[mask, "lock_time"]
        b.loc[mask, "prev_dispatch"] = b.loc[mask, "p_min"]
        b.loc[mask, "min_cap"] = b.loc[mask, "p_min"]
        b.loc[mask, "max_cap"] = b.loc[mask, "prev_dispatch"] + b.loc[mask, "ramp_hour"]
        shutdown_cost = b[mask]["sc"].sum()
        b = b.iloc[:, :-1]
        self.shutdown_costs += shutdown_cost
        self.bids_now = b.copy()
        self.__log_market_log('shutting_down_marginal_unit')

    def __startup_units(self):
        b = self.bids_now.copy()
        mask = (b["dispatch"] > 0) & (b["online"] == False)
        startup_cost = b[mask]["sc"].sum()
        b.loc[mask, "online"] = True
        self.startup_costs += startup_cost
        self.bids_now = b.copy()

    def __calculate_cost_price(self):
        b = self.bids_now.copy()
        self.mcp = b[b['dispatch'] > 0]['mc'].max()
        self.energy_costs = b[b['dispatch'] > 0]['dispatch'].sum() * self.mcp
        self.system_costs = self.energy_costs + self.shutdown_costs + self.startup_costs

    def __record_logs(self):
        b = self.bids_now.copy()
        
        b['schedule'] = self.schedule
        b['hour'] = self.hour
        b['load'] = self.load
        b['mcp'] = self.mcp
        b['energy_costs'] = self.energy_costs
        b['shutdown_costs'] = self.shutdown_costs
        b['startup_costs'] = self.startup_costs
        b['system_cost'] = self.system_costs
        self.logs = pd.concat([self.logs,b])

        standard_cols = [
            'id','p_max','p_min','mc','sc','lock_time',"ramp_hour",'online','locked','prev_dispatch','max_cap','min_cap','type','dispatch'
        ]
            
        # rewrite bids
        self.bids_now = b[standard_cols].copy()
        self.bids = self.bids_now.copy()

In [46]:
# New Market Model

class Market_2():
    def __init__(self, initial_bids_state:pd.DataFrame, load_schedule:pd.DataFrame):
        self.bids = initial_bids_state
        self.load_schedule = load_schedule
        self.logs = pd.DataFrame()
        self.market_logs = pd.DataFrame()
        self.market_log_slots = None
        self.undersupply = False
        
        # to normalize data type
        self.bids["locked"] = self.bids["locked"].astype("int32")
        self.bids["lock_time"] = self.bids["lock_time"].astype("int32")
        
    def start(self):
        
        # Main loop over each hour begins here
        
        for row in self.load_schedule.itertuples(index=False):
            self.load = row.load
            self.schedule = row.schedule
            self.hour = row.hour
            self.shutdown_costs = 0
            self.startup_costs = 0
            self.energy_costs = 0
            
            self.event_squence = 0
            self.logging = False
            if self.market_log_slots != None:
                if (self.schedule, self.hour) in self.market_log_slots:
                    self.logging = True
            
            # Enter into the market cycle
            self.__market_cycle()
            if self.undersupply:
                print('Load exceeds capacity!')
                return
            self.__startup_units()
            self.__calculate_cost_price()
            self.__record_logs() 
    
    def __prepare_new_bids(self):
        b = self.bids_now.copy()
        b["prev_dispatch"] = np.where(b["dispatch"] > 0, b["dispatch"], b["p_min"])
        b["max_cap"] = np.where(
            b["prev_dispatch"] + b["ramp_hour"] <= b["p_max"],
            b["prev_dispatch"] + b["ramp_hour"],
            b["p_max"],
        )
        b["min_cap"] = np.where(
            b["prev_dispatch"] - b["ramp_hour"] >= b["p_min"],
            b["prev_dispatch"] - b["ramp_hour"],
            b["p_min"],
        )        
        self.bids_now = b.copy()
    
    def activate_market_logs(self, from_sch_hour:tuple, to_sch_hour:tuple):
        f = from_sch_hour
        t = to_sch_hour
        market_log_slots = []
        for schedule in range(f[0], t[0] + 1):
            start_hour = f[1] if schedule == f[0] else 0
            end_hour = t[1] if schedule == t[0] else 23
            for hour in range(start_hour, end_hour + 1):
                market_log_slots.append((schedule, hour))
        self.market_log_slots = market_log_slots
        
    def __log_market_log(self, event_label='',final=False):
        if not self.logging:
            return
        
        b = self.bids_now.copy()
        b['event'] = event_label
        b['sequence'] = self.event_squence
        b['schedule'] = self.schedule
        b['hour'] = self.hour
        self.event_squence += 1
        self.market_logs = pd.concat([self.market_logs, b])

    def __market_cycle(self):
        
        self.bids_now = self.bids.copy()
        self.__log_market_log('start')
        
        self.__prepare_new_bids()        
        self.__update_locks()
        self.__order_by_merit()
        if self.undersupply:
            return
        self.__check_constraints()
        
        # market operator / system operator must 
        # decide what to do with constraints
        
        # 1. shutdown non-preferred unit
        if self.constraints_found:   
            self.__shutdown_nonpreferred(preferred_ids=[]) # try 35,34
            self.__order_by_merit()
            if self.undersupply:
                return
            self.__check_constraints()
            

            # 2. curtail in favor of marginal unit
            if self.constraints_found:
                self.__curtailment_protocol()
                
                # 3. if that's not feasible, shutdown marginal until no more constraint
                if not self.curtailment_feasible:
                    print(self.curtailment_feasible)
                    while self.constraints_found:
                        self.__shutdown_marginal_only()
                        self.__order_by_merit()
                        if self.undersupply:
                            return
                        self.__check_constraints()

        # clean up temporary columns
        self.bids_now = self.bids_now.drop(columns=['constrained', 'curtail', 'redispatch'], errors='ignore')
  
    def __update_locks(self):
        b = self.bids_now.copy()
        b["locked"] = np.where(b["locked"] - 1 >= 0, b["locked"] - 1, b["locked"])
        self.bids_now = b
        self.__log_market_log('updating_locks')
        
    def __order_by_merit(self):
        load = self.load
        b = self.bids_now.copy()
        b = b.sort_values(["locked", "mc"])

        self.undersupply = False

        # Check if load exceeds available capacity
        # If so, break the market cycle
        
        if load > b[b["locked"] == 0]["max_cap"].sum():
            self.undersupply = True
            return

        dispatches = []
        leftovers = []

        leftover = load
        for row in b.itertuples(index=False):
            dispatch = min(leftover, row.max_cap)
            dispatches.append(dispatch)
            leftover = leftover - dispatch
            leftovers.append(leftover)

            if leftover <= 0:
                pass

        b["dispatch"] = dispatches
        b["leftover"] = leftovers
                
        self.bids_now = b
        self.__log_market_log('merit_ordering')
    
    def __check_constraints(self):
        b = self.bids_now.copy()
        active_cond = b["dispatch"] > 0
        b["constrained"] = 0
        b["constrained"] = np.where(
            (b["dispatch"] < b["p_min"]) & active_cond, 1, b["constrained"]
        )
        b["constrained"] = np.where(
            (b["dispatch"] > b["p_max"]) & active_cond, 1, b["constrained"]
        )
        b["constrained"] = np.where((b["locked"] > 0) & active_cond, 1, b["constrained"])
        b["constrained"] = np.where(
            (b["online"] == True) & ~(active_cond), 1, b["constrained"]
        )
        b['constrained'] = np.where(
            b["online"] == False, 0, b["constrained"]
        )
        
        if b["constrained"].sum() > 0:
            self.constraints_found = True
        else:
            self.constraints_found = False
            
        self.bids_now = b.copy()
        self.__log_market_log('checking_constraint')

    def __shutdown_nonpreferred(self, preferred_ids=[]):
        
        b = self.bids_now.copy()
        mask = (b['constrained'] == 1) & (~b['id'].isin(preferred_ids) & (b['dispatch'] == 0))
        b.loc[mask, "online"] = False
        b.loc[mask, "locked"] = b.loc[mask, "lock_time"]
        b.loc[mask, "prev_dispatch"] = b.loc[mask, "p_min"]
        b.loc[mask, "min_cap"] = b.loc[mask, "p_min"]
        b.loc[mask, "max_cap"] = b.loc[mask, "prev_dispatch"] + b.loc[mask, "ramp_hour"]
        b['max_cap'] = np.where(b['max_cap'] > b['p_max'], b['p_max'], b['max_cap'])
        shutdown_cost = b[mask]["sc"].sum()
        self.shutdown_costs += shutdown_cost
        self.bids_now = b.copy()
        self.__log_market_log('shutting_down_nonpreferred')

    def __curtailment_protocol(self):
        load = self.load
        b = self.bids_now.copy()
        b["curtail"] = np.where((b["dispatch"] > 0) & (b["constrained"] == 0), 1, 0)
        retained_power = b[b["constrained"] == 1]["p_min"].sum()
        final_curtailed_power = load - retained_power
        initial_power_to_curtail = b[b["curtail"] == 1]["dispatch"].sum()
        if initial_power_to_curtail != 0:
            curtailment_factor = final_curtailed_power / initial_power_to_curtail
        else:
            curtailment_factor = 0
            
        b["redispatch"] = np.where(
            b["dispatch"] * curtailment_factor >= b["p_min"],
            b["dispatch"] * curtailment_factor,
            b["p_min"],
        )
        b["redispatch"] = np.where(
            (b["constrained"] == 0) & (b["curtail"] == 0), 0, b["redispatch"]
        )
        
        self.curtailment_feasible = None

        total_redispatch = b["redispatch"].sum()
        tol = 1e-6  # prevents floating point comparison errors e.g. 1751.4699999999998 < 1751.47
        
        if total_redispatch + tol >= load :
            b["dispatch"] = b["redispatch"]
            self.curtailment_feasible = True
            self.bids_now = b.copy()
        else:
            b["dispatch"] = b["redispatch"]
            self.curtailment_feasible = False
            self.bids_now = b.copy()
            
        self.__log_market_log('attempting_curtailment')

    def __shutdown_marginal_only(self):
        b = self.bids_now.copy()
        mask = (b['online'] == True) & (b['online'].shift(-1) == False)
        b.loc[mask, "online"] = False
        b.loc[mask, "locked"] = b.loc[mask, "lock_time"]
        b.loc[mask, "prev_dispatch"] = b.loc[mask, "p_min"]
        b.loc[mask, "min_cap"] = b.loc[mask, "p_min"]
        b.loc[mask, "max_cap"] = b.loc[mask, "prev_dispatch"] + b.loc[mask, "ramp_hour"]
        shutdown_cost = b[mask]["sc"].sum()
        b = b.iloc[:, :-1]
        self.shutdown_costs += shutdown_cost
        self.bids_now = b.copy()
        self.__log_market_log('shutting_down_marginal_unit')

    def __startup_units(self):
        b = self.bids_now.copy()
        mask = (b["dispatch"] > 0) & (b["online"] == False)
        startup_cost = b[mask]["sc"].sum()
        b.loc[mask, "online"] = True
        self.startup_costs += startup_cost
        self.bids_now = b.copy()

    def __calculate_cost_price(self):
        b = self.bids_now.copy()
        self.mcp = b[b['dispatch'] > 0]['mc'].max()
        self.energy_costs = b[b['dispatch'] > 0]['dispatch'].sum() * self.mcp
        self.system_costs = self.energy_costs + self.shutdown_costs + self.startup_costs

    def __record_logs(self):
        b = self.bids_now.copy()
        
        b['schedule'] = self.schedule
        b['hour'] = self.hour
        b['load'] = self.load
        b['mcp'] = self.mcp
        b['energy_costs'] = self.energy_costs
        b['shutdown_costs'] = self.shutdown_costs
        b['startup_costs'] = self.startup_costs
        b['system_cost'] = self.system_costs
        self.logs = pd.concat([self.logs,b])

        standard_cols = [
            'id','p_max','p_min','mc','sc','lock_time',"ramp_hour",'online','locked','prev_dispatch','max_cap','min_cap','type','dispatch'
        ]
            
        # rewrite bids
        self.bids_now = b[standard_cols].copy()
        self.bids = self.bids_now.copy()

In [47]:
s = df.iloc[:3]
m = Market(original_bids,s)
m.start()
next_bid = m.bids_now.copy()